In [1]:
import pandas as pd

# 데이터

In [ ]:
data=pd.read_csv("행정코드매칭데이터.csv", encoding="utf-8")

In [3]:
data.head()

,address,적용유형,시설유형,점수
0,경상북도 경주시 노동동,영농상속형,교육,10
1,경상북도 경주시 노동동,영농상속형,교통,6
2,경상북도 경주시 노동동,영농상속형,농업_매물,8
3,경상북도 경주시 노동동,영농상속형,농업_지원,10
4,경상북도 경주시 노동동,영농상속형,서비스,10


# 법정동코드 데이터

In [4]:
code = pd.read_csv("법정동코드 전체자료.txt", delimiter="\t", encoding="cp949")
code.drop(columns=["폐지여부"], inplace=True)
code.head()

,법정동코드,법정동명
0,1100000000,서울특별시
1,1111000000,서울특별시 종로구
2,1111010100,서울특별시 종로구 청운동
3,1111010200,서울특별시 종로구 신교동
4,1111010300,서울특별시 종로구 궁정동


In [5]:
code['법정동코드_8자리'] = code['법정동코드'].astype(str).str[:8]
code.head()

,법정동코드,법정동명,법정동코드_8자리
0,1100000000,서울특별시,11000000
1,1111000000,서울특별시 종로구,11110000
2,1111010100,서울특별시 종로구 청운동,11110101
3,1111010200,서울특별시 종로구 신교동,11110102
4,1111010300,서울특별시 종로구 궁정동,11110103


읍면동까지 있는경우 - 8자리  
리까지 있는경우 - 10자리

# 주소 분할

In [6]:
# 네개로 분할하는 함수
def split_full_address(addr):
    if isinstance(addr, str):  # 문자열일 때만 처리
        parts = addr.strip().split()
        if len(parts) >= 4:
            return pd.Series([parts[0], parts[1], parts[2], " ".join(parts[3:])])
        elif len(parts) == 3:
            return pd.Series([parts[0], parts[1], parts[2], pd.NA]) #읍면동까지
        elif len(parts) == 2:
            return pd.Series([parts[0], parts[1], pd.NA, pd.NA]) #시군구까지
        else:
            return pd.Series([addr, pd.NA, pd.NA, pd.NA]) #나머지주소만 (기타)
    else:
        return pd.Series([pd.NA, pd.NA, pd.NA, pd.NA]) #주소가 없을 경우

In [7]:
data[['시도', '시군구', '읍면동', '리']] = data['address'].apply(split_full_address)
data.head()

,address,적용유형,시설유형,점수,시도,시군구,읍면동,리
0,경상북도 경주시 노동동,영농상속형,교육,10,경상북도,경주시,노동동,<NA>
1,경상북도 경주시 노동동,영농상속형,교통,6,경상북도,경주시,노동동,<NA>
2,경상북도 경주시 노동동,영농상속형,농업_매물,8,경상북도,경주시,노동동,<NA>
3,경상북도 경주시 노동동,영농상속형,농업_지원,10,경상북도,경주시,노동동,<NA>
4,경상북도 경주시 노동동,영농상속형,서비스,10,경상북도,경주시,노동동,<NA>


In [8]:
data.isna().sum()

address      0
적용유형         0
시설유형         0
점수           0
시도           0
시군구          0
읍면동          0
리          222
dtype: int64

In [10]:
code[['시도', '시군구', '읍면동', '리']] = code['법정동명'].apply(split_full_address)
code.head()

,법정동코드,법정동명,법정동코드_8자리,시도,시군구,읍면동,리
0,1100000000,서울특별시,11000000,서울특별시,<NA>,<NA>,<NA>
1,1111000000,서울특별시 종로구,11110000,서울특별시,종로구,<NA>,<NA>
2,1111010100,서울특별시 종로구 청운동,11110101,서울특별시,종로구,청운동,<NA>
3,1111010200,서울특별시 종로구 신교동,11110102,서울특별시,종로구,신교동,<NA>
4,1111010300,서울특별시 종로구 궁정동,11110103,서울특별시,종로구,궁정동,<NA>


"시도" 오류 수정

In [11]:
code["시도"].unique()

array(['서울특별시', '부산직할시', '대구직할시', '인천직할시', '광주직할시', '대전직할시', '부산광역시',
       '대구광역시', '인천광역시', '광주광역시', '대전광역시', '울산광역시', '세종특별자치시', '경기도',
       '강원도', '충청북도', '충청남도', '전라북도', '전라남도', '경상북도', '경상남도', '제주도',
       '제주특별자치도', '강원특별자치도', '전북특별자치도'], dtype=object)

In [12]:
code.replace({"시도":{"전북특별자치도":"전라북도"}}, inplace=True)

In [13]:
code["시도"].unique()

array(['서울특별시', '부산직할시', '대구직할시', '인천직할시', '광주직할시', '대전직할시', '부산광역시',
       '대구광역시', '인천광역시', '광주광역시', '대전광역시', '울산광역시', '세종특별자치시', '경기도',
       '강원도', '충청북도', '충청남도', '전라북도', '전라남도', '경상북도', '경상남도', '제주도',
       '제주특별자치도', '강원특별자치도'], dtype=object)

In [14]:
code.shape

(49859, 7)

# 법정동 코드 빈집주소 매칭

## 1. 읍면동까지

In [15]:
temp_dong=data[data["리"].isna()]
temp_dong.head()

,address,적용유형,시설유형,점수,시도,시군구,읍면동,리
0,경상북도 경주시 노동동,영농상속형,교육,10,경상북도,경주시,노동동,<NA>
1,경상북도 경주시 노동동,영농상속형,교통,6,경상북도,경주시,노동동,<NA>
2,경상북도 경주시 노동동,영농상속형,농업_매물,8,경상북도,경주시,노동동,<NA>
3,경상북도 경주시 노동동,영농상속형,농업_지원,10,경상북도,경주시,노동동,<NA>
4,경상북도 경주시 노동동,영농상속형,서비스,10,경상북도,경주시,노동동,<NA>


In [16]:
code_dong=code[code["리"].isna()]
code_dong.head()

,법정동코드,법정동명,법정동코드_8자리,시도,시군구,읍면동,리
0,1100000000,서울특별시,11000000,서울특별시,<NA>,<NA>,<NA>
1,1111000000,서울특별시 종로구,11110000,서울특별시,종로구,<NA>,<NA>
2,1111010100,서울특별시 종로구 청운동,11110101,서울특별시,종로구,청운동,<NA>
3,1111010200,서울특별시 종로구 신교동,11110102,서울특별시,종로구,신교동,<NA>
4,1111010300,서울특별시 종로구 궁정동,11110103,서울특별시,종로구,궁정동,<NA>


In [18]:
merge_dong=pd.merge(temp_dong, code_dong, how="left", on=['시도','시군구','읍면동', '리'])
merge_dong=merge_dong[["address", '적용유형', '시설유형', '점수', "시도", "시군구", "읍면동", "리", "법정동코드_8자리"]]
merge_dong.rename(columns={"법정동코드_8자리":"법정동코드"}, inplace=True)
merge_dong.head()

,address,적용유형,시설유형,점수,시도,시군구,읍면동,리,법정동코드
0,경상북도 경주시 노동동,영농상속형,교육,10,경상북도,경주시,노동동,<NA>,47130106
1,경상북도 경주시 노동동,영농상속형,교통,6,경상북도,경주시,노동동,<NA>,47130106
2,경상북도 경주시 노동동,영농상속형,농업_매물,8,경상북도,경주시,노동동,<NA>,47130106
3,경상북도 경주시 노동동,영농상속형,농업_지원,10,경상북도,경주시,노동동,<NA>,47130106
4,경상북도 경주시 노동동,영농상속형,서비스,10,경상북도,경주시,노동동,<NA>,47130106


In [19]:
merge_dong.isna().sum()

address      0
적용유형         0
시설유형         0
점수           0
시도           0
시군구          0
읍면동          0
리          222
법정동코드       30
dtype: int64

데이터의 손실이 있는건 수동으로 추가ㅣ

In [22]:
merge_dong[merge_dong["법정동코드"].isna()]["address"].unique()

array(['전라북도 전주덕진구 우아동3가 ', '전라북도 전주덕진구 호성동1가 '], dtype=object)

In [23]:
merge_dong.loc[merge_dong["address"] == "전라북도 전주덕진구 우아동3가 ", "법정동코드"] = "45113115"
merge_dong.loc[merge_dong["address"] == "전라북도 전주덕진구 호성동1가 ", "법정동코드"] = "45113116"

In [24]:
merge_dong.isna().sum()

address      0
적용유형         0
시설유형         0
점수           0
시도           0
시군구          0
읍면동          0
리          222
법정동코드        0
dtype: int64

## 2. 리까지

In [25]:
temp_li = data[data["리"].notna()]
temp_li.head()

,address,적용유형,시설유형,점수,시도,시군구,읍면동,리
48,경상북도 경주시 안강읍 안강리,청년창농형,교육,10,경상북도,경주시,안강읍,안강리
49,경상북도 경주시 안강읍 안강리,청년창농형,교통,9,경상북도,경주시,안강읍,안강리
50,경상북도 경주시 안강읍 안강리,청년창농형,농업_매물,10,경상북도,경주시,안강읍,안강리
51,경상북도 경주시 안강읍 안강리,청년창농형,농업_지원,9,경상북도,경주시,안강읍,안강리
52,경상북도 경주시 안강읍 안강리,청년창농형,서비스,8,경상북도,경주시,안강읍,안강리


In [26]:
code_li=code[code["리"].notna()]
code_li.head()

,법정동코드,법정동명,법정동코드_8자리,시도,시군구,읍면동,리
2835,2671025021,부산광역시 기장군 기장읍 동부리,26710250,부산광역시,기장군,기장읍,동부리
2836,2671025022,부산광역시 기장군 기장읍 교리,26710250,부산광역시,기장군,기장읍,교리
2837,2671025023,부산광역시 기장군 기장읍 신천리,26710250,부산광역시,기장군,기장읍,신천리
2838,2671025024,부산광역시 기장군 기장읍 죽성리,26710250,부산광역시,기장군,기장읍,죽성리
2839,2671025025,부산광역시 기장군 기장읍 서부리,26710250,부산광역시,기장군,기장읍,서부리


In [33]:
merge_li=pd.merge(temp_li, code_li, how="left", on=['시도','시군구','읍면동', '리'])
merge_li=merge_li[["address", '적용유형', '시설유형', '점수', "시도", "시군구", "읍면동", "리", "법정동코드"]]
merge_li.head()

,address,적용유형,시설유형,점수,시도,시군구,읍면동,리,법정동코드
0,경상북도 경주시 안강읍 안강리,청년창농형,교육,10,경상북도,경주시,안강읍,안강리,4713025333
1,경상북도 경주시 안강읍 안강리,청년창농형,교통,9,경상북도,경주시,안강읍,안강리,4713025333
2,경상북도 경주시 안강읍 안강리,청년창농형,농업_매물,10,경상북도,경주시,안강읍,안강리,4713025333
3,경상북도 경주시 안강읍 안강리,청년창농형,농업_지원,9,경상북도,경주시,안강읍,안강리,4713025333
4,경상북도 경주시 안강읍 안강리,청년창농형,서비스,8,경상북도,경주시,안강읍,안강리,4713025333


In [34]:
merge_li.isna().sum()

address    0
적용유형       0
시설유형       0
점수         0
시도         0
시군구        0
읍면동        0
리          0
법정동코드      0
dtype: int64

## 결과 합산

In [35]:
result=pd.concat([merge_dong, merge_li], axis=0)
result.reset_index(drop=True, inplace=True)
result.head()

,address,적용유형,시설유형,점수,시도,시군구,읍면동,리,법정동코드
0,경상북도 경주시 노동동,영농상속형,교육,10,경상북도,경주시,노동동,<NA>,47130106
1,경상북도 경주시 노동동,영농상속형,교통,6,경상북도,경주시,노동동,<NA>,47130106
2,경상북도 경주시 노동동,영농상속형,농업_매물,8,경상북도,경주시,노동동,<NA>,47130106
3,경상북도 경주시 노동동,영농상속형,농업_지원,10,경상북도,경주시,노동동,<NA>,47130106
4,경상북도 경주시 노동동,영농상속형,서비스,10,경상북도,경주시,노동동,<NA>,47130106


In [37]:
result['법정동코드'] = result['법정동코드'].astype('object')

In [38]:
result.dtypes

address    object
적용유형       object
시설유형       object
점수          int64
시도         object
시군구        object
읍면동        object
리          object
법정동코드      object
dtype: object

# 파일 생성

In [39]:
result.to_csv("태블로용_주소_귀농유형_클러스터.csv", index=False, encoding="utf-8-sig")